# Bakehouse Deep Analytics

**Dataset:** `samples.bakehouse (all tables)`

**Difficulty:** Hard

**Topics:** multi-table joins, customer LTV, cohort, PII masking, window functions, data quality

In [ ]:
from pyspark.sql import functions as F, types as T
from pyspark.sql import Window

transactions = spark.table("samples.bakehouse.sales_transactions")
customers    = spark.table("samples.bakehouse.sales_customers")
franchises   = spark.table("samples.bakehouse.sales_franchises")
suppliers    = spark.table("samples.bakehouse.sales_suppliers")
reviews      = spark.table("samples.bakehouse.media_customer_reviews")
gold_reviews = spark.table("samples.bakehouse.media_gold_reviews_chunked")

## Problem 1

**Full Transaction Enrichment.**

Join all four sales tables (transactions + customers + franchises + suppliers) to create a flat analytical table. For each transaction expose key fields from every dimension table so that downstream BI tools need only query a single denormalised dataset.

Business context: The finance team needs a single enriched dataset that combines transactional, customer, franchise, and supplier data for month-end reporting without requiring complex joins each time.

**Expected output columns:** `transactionID`, `dateTime`, `product`, `quantity`, `totalPrice`, `customerID`, `customer_country`, `franchiseName`, `franchise_country`, `franchise_size`

In [ ]:
# Problem 1 — write your solution here
# Assign your result to: result_1
# then -> suppliers on supplierID; select and rename columns as needed

result_1 = None  # replace this

In [ ]:
# ── Tests for Problem 1 ──────────────────────────────────────────
assert result_1 is not None, "result_1 is None — did you assign your DataFrame?"
assert hasattr(result_1, 'columns'), "result_1 must be a Spark DataFrame"
cols = [c.lower() for c in result_1.columns]
for col in ['transactionid', 'datetime', 'product', 'quantity', 'totalprice',
            'customerid', 'customer_country', 'franchisename', 'franchise_country', 'franchise_size']:
    assert col in cols, f"Missing column: {col}"
cnt = result_1.count()
assert cnt > 0, f"Expected rows > 0, got {cnt}"
null_price = result_1.filter(F.col('totalPrice').isNull()).count()
assert null_price == 0, f"Expected no null totalPrice values, got {null_price}"
print(f"Problem 1 passed ✓  ({cnt} rows)")

## Problem 2

**Customer Lifetime Value (LTV).**

Per customer compute: total spend, number of transactions, average order value, days between first and last purchase (`tenure_days`), and annualised LTV = `total_spend / tenure_days * 365`. Join with the customers table to include `first_name` and `last_name`. Customers with only one purchase have tenure_days = 1 (avoid division by zero).

Business context: LTV ranking allows the marketing team to identify high-value customers for exclusive loyalty offers and early access to new products, maximising retention ROI.

**Expected output columns:** `customerID`, `first_name`, `last_name`, `total_spend`, `transaction_count`, `avg_order_value`, `tenure_days`, `annualised_ltv`

In [ ]:
# Problem 2 — write your solution here
# Assign your result to: result_2
# Then compute tenure_days = datediff(max_date, min_date), handle zero with F.greatest(..., F.lit(1))
# annualised_ltv = total_spend / tenure_days * 365

result_2 = None  # replace this

In [ ]:
# ── Tests for Problem 2 ──────────────────────────────────────────
assert result_2 is not None, "result_2 is None — did you assign your DataFrame?"
assert hasattr(result_2, 'columns'), "result_2 must be a Spark DataFrame"
cols = [c.lower() for c in result_2.columns]
for col in ['customerid', 'first_name', 'last_name', 'total_spend',
            'transaction_count', 'avg_order_value', 'tenure_days', 'annualised_ltv']:
    assert col in cols, f"Missing column: {col}"
cnt = result_2.count()
assert cnt > 0, f"Expected rows > 0, got {cnt}"
assert result_2.agg(F.min('total_spend')).collect()[0][0] > 0, "total_spend must be positive"
assert result_2.agg(F.min('tenure_days')).collect()[0][0] >= 1, "tenure_days must be >= 1"
assert result_2.agg(F.min('annualised_ltv')).collect()[0][0] > 0, "annualised_ltv must be positive"
print(f"Problem 2 passed ✓  ({cnt} rows)")

## Problem 3

**Revenue Cohort Analysis.**

Group customers by their first purchase month (`cohort_month`). For each cohort show the total revenue generated in each subsequent period expressed as months since first purchase (`months_since_first`). Include the cohort size (distinct customers in that cohort).

Business context: Cohort revenue analysis reveals whether newer customer cohorts are monetising at a better rate than older ones, helping the product team assess the impact of loyalty-programme changes.


**Expected output columns:** `cohort_month`, `months_since_first`, `cohort_size`, `revenue`

In [ ]:
# Problem 3 — write your solution here
# Assign your result to: result_3
#   1. Compute each customer's first purchase month: groupBy customerID -> min(dateTime) -> date_trunc month
#   2. Join back to transactions
#   3. Compute purchase_month = date_trunc('month', dateTime)
#   4. months_since_first = F.round(F.months_between(purchase_month, cohort_month)).cast('int')
#   5. groupBy cohort_month + months_since_first -> sum revenue, count distinct customers

result_3 = None  # replace this

In [ ]:
# ── Tests for Problem 3 ──────────────────────────────────────────
assert result_3 is not None, "result_3 is None — did you assign your DataFrame?"
assert hasattr(result_3, 'columns'), "result_3 must be a Spark DataFrame"
cols = [c.lower() for c in result_3.columns]
for col in ['cohort_month', 'months_since_first', 'cohort_size', 'revenue']:
    assert col in cols, f"Missing column: {col}"
cnt = result_3.count()
assert cnt > 0, f"Expected rows > 0, got {cnt}"
min_months = result_3.agg(F.min('months_since_first')).collect()[0][0]
assert min_months == 0, f"months_since_first should start at 0, got {min_months}"
assert result_3.agg(F.min('revenue')).collect()[0][0] > 0, "revenue must be positive"
assert result_3.agg(F.min('cohort_size')).collect()[0][0] > 0, "cohort_size must be positive"
print(f"Problem 3 passed ✓  ({cnt} rows)")

## Problem 4

**Discount Detection.**

Find all transactions where `totalPrice < unitPrice * quantity`, indicating that a discount was applied at point of sale. Compute the discount amount (`expected_price - totalPrice`) and discount percentage (`discount_amount / expected_price * 100`).

Business context: Unauthorised discounting erodes margins. The finance team needs a report of all discounted transactions to audit whether discounts were applied by authorised personnel and whether the discount levels comply with policy.

**Expected output columns:** `transactionID`, `customerID`, `franchiseID`, `product`, `unitPrice`, `quantity`, `totalPrice`, `expected_price`, `discount_amount`, `discount_pct`

In [ ]:
# Problem 4 — write your solution here
# Assign your result to: result_4
# then compute discount_amount and discount_pct

result_4 = None  # replace this

In [ ]:
# ── Tests for Problem 4 ──────────────────────────────────────────
assert result_4 is not None, "result_4 is None — did you assign your DataFrame?"
assert hasattr(result_4, 'columns'), "result_4 must be a Spark DataFrame"
cols = [c.lower() for c in result_4.columns]
for col in ['transactionid', 'customerid', 'franchiseid', 'product',
            'expected_price', 'discount_amount', 'discount_pct']:
    assert col in cols, f"Missing column: {col}"
cnt = result_4.count()
assert cnt > 0, f"Expected rows > 0, got {cnt}"
assert result_4.agg(F.min('discount_amount')).collect()[0][0] > 0, "discount_amount must be positive"
assert result_4.agg(F.min('discount_pct')).collect()[0][0] > 0, "discount_pct must be positive"
assert result_4.agg(F.max('discount_pct')).collect()[0][0] <= 100, "discount_pct cannot exceed 100"
print(f"Problem 4 passed ✓  ({cnt} rows)")

## Problem 5

**Franchise Performance Scorecard.**

For each franchise compute: total revenue, transaction count, unique customers, average transaction value, and review engagement score (join `media_customer_reviews` on `franchiseID`). The bakehouse reviews table has only text reviews (no numeric rating), so use **review count per franchise** as the engagement metric — alias this as `avg_review_score`. Then rank franchises by a composite score = sum of their `total_revenue` rank and `avg_review_score` rank (lower = better). Include franchise name from the franchises table.

Business context: The operations director needs a single scorecard that balances commercial performance (revenue) with customer engagement (review volume) to identify franchises that need operational support.


**Expected output columns:** `franchiseID`, `franchiseName`, `total_revenue`, `transaction_count`, `unique_customers`, `avg_review_score`, `composite_rank`

In [ ]:
# Problem 5 — write your solution here
# Assign your result to: result_5
#   1. Aggregate transactions by franchiseID: sum(totalPrice), count, countDistinct(customerID), avg
#   2. Aggregate reviews by franchiseID: avg review score (reviews table has 'review' as text —
#      use a proxy like review count, or cast the review text; use new_id or franchiseID)
#   3. Join franchises for name
#   4. Compute rank on revenue (desc) and rank on review_score (desc)
#   5. composite_score = rev_rank + review_rank; rank on composite_score ascending

result_5 = None  # replace this

In [ ]:
# ── Tests for Problem 5 ──────────────────────────────────────────
assert result_5 is not None, "result_5 is None — did you assign your DataFrame?"
assert hasattr(result_5, 'columns'), "result_5 must be a Spark DataFrame"
cols = [c.lower() for c in result_5.columns]
for col in ['franchiseid', 'franchisename', 'total_revenue', 'transaction_count',
            'unique_customers', 'avg_review_score', 'composite_rank']:
    assert col in cols, f"Missing column: {col}"
cnt = result_5.count()
assert cnt > 0, f"Expected rows > 0, got {cnt}"
min_rank = result_5.agg(F.min('composite_rank')).collect()[0][0]
assert min_rank == 1, f"Minimum composite_rank should be 1, got {min_rank}"
assert result_5.agg(F.min('total_revenue')).collect()[0][0] > 0, "total_revenue must be positive"
print(f"Problem 5 passed ✓  ({cnt} rows)")

## Problem 6

**Product Affinity.**

Find pairs of products that are frequently bought together within the same transaction (same `transactionID`). Count the number of transactions in which each product pair co-occurs. Ensure no duplicate pairs by enforcing `product_a < product_b` alphabetically (self-join on transactionID).

Business context: Product affinity analysis powers the 'frequently bought together' feature on the in-store ordering kiosk, which the marketing team expects will increase average basket size by 8–12%.


**Expected output columns:** `product_a`, `product_b`, `co_occurrence_count`

In [ ]:
# Problem 6 — write your solution here
# Assign your result to: result_6
#   t1 = transactions.select('transactionID', F.col('product').alias('product_a'))
#   t2 = transactions.select('transactionID', F.col('product').alias('product_b'))
#   joined = t1.join(t2, on='transactionID').filter(F.col('product_a') < F.col('product_b'))
#   result_6 = joined.groupBy('product_a', 'product_b').count().withColumnRenamed('count','co_occurrence_count')
#              .orderBy(F.desc('co_occurrence_count'))

result_6 = None  # replace this

In [ ]:
# ── Tests for Problem 6 ──────────────────────────────────────────
assert result_6 is not None, "result_6 is None — did you assign your DataFrame?"
assert hasattr(result_6, 'columns'), "result_6 must be a Spark DataFrame"
cols = [c.lower() for c in result_6.columns]
for col in ['product_a', 'product_b', 'co_occurrence_count']:
    assert col in cols, f"Missing column: {col}"
cnt = result_6.count()
assert cnt > 0, f"Expected rows > 0, got {cnt}"
# Verify no duplicate pairs (all product_a < product_b)
bad_pairs = result_6.filter(F.col('product_a') >= F.col('product_b')).count()
assert bad_pairs == 0, f"Found {bad_pairs} rows where product_a >= product_b — pairs should be deduplicated"
assert result_6.agg(F.min('co_occurrence_count')).collect()[0][0] > 0, "co_occurrence_count must be positive"
print(f"Problem 6 passed ✓  ({cnt} rows)")

## Problem 7

**PII Data Masking.**

Build an operational dashboard dataset with all PII masked using the following rules:
- `first_name` / `last_name`: fully replace with `***`
- `email_address`: mask the username (keep only the first character, replace the rest with `*`), keep the full domain. E.g. `john.doe@gmail.com` → `j*******@gmail.com`
- `phone_number`: keep only the last 3 digits, replace the rest with `***-***-`
- `cardNumber`: keep last 4 digits as a string, replace the rest with `****-****-****-`

Business context: GDPR and PCI-DSS compliance requires that any dataset shared outside the secure data warehouse has PII and payment data masked before export.


**Expected output columns:** `date`, `dateTime`, `franchiseName`, `firstName`, `lastName`, `emailAddress`, `phoneNumber`, `city`, `country`, `product`, `quantity`, `unitPrice`, `totalPrice`, `methodOfPayment`, `cardNumber`

In [ ]:
# Problem 7 — write your solution here
# Assign your result to: result_7
#   Join transactions + customers + franchises
#   firstName = F.lit('***')
#   lastName = F.lit('***')
#   email masking: F.concat(F.substring('email_address', 1, 1),
#                           F.regexp_replace(F.split('email_address', '@')[0], '.', '*'),
#                           F.lit('@'), F.split('email_address', '@')[1])
#   phone masking: F.concat(F.lit('***-***-'), F.substring('phone_number', -3, 3))
#   cardNumber masking: F.concat(F.lit('****-****-****-'), (F.col('cardNumber') % 10000).cast('string'))

result_7 = None  # replace this

In [ ]:
# ── Tests for Problem 7 ──────────────────────────────────────────
assert result_7 is not None, "result_7 is None — did you assign your DataFrame?"
assert hasattr(result_7, 'columns'), "result_7 must be a Spark DataFrame"
cols = [c.lower() for c in result_7.columns]
for col in ['datetime', 'franchisename', 'firstname', 'lastname',
            'emailaddress', 'phonenumber', 'product', 'totalprice']:
    assert col in cols, f"Missing column: {col}"
cnt = result_7.count()
assert cnt > 0, f"Expected rows > 0, got {cnt}"
# All first names should be masked
unmasked = result_7.filter(F.col('firstName') != '***').count()
assert unmasked == 0, f"Found {unmasked} unmasked first names — all should be '***'"
# Email should contain @ symbol
no_at = result_7.filter(~F.col('emailAddress').contains('@')).count()
assert no_at == 0, f"Found {no_at} email addresses without '@' symbol"
print(f"Problem 7 passed ✓  ({cnt} rows)")

## Problem 8

**Data Quality Report.**

For each of the 4 sales tables (`sales_transactions`, `sales_customers`, `sales_franchises`, `sales_suppliers`) compute a data quality summary: table name, total row count, number of duplicate primary keys, and a list of column names that have any null values.

Business context: Before onboarding a new data vendor or migration, the data engineering team requires a structured data quality report to quantify completeness and uniqueness so that issues can be fixed upstream.


**Expected output columns:** `table_name`, `row_count`, `null_columns`, `duplicate_pk_count`

In [ ]:
# Problem 8 — write your solution here
# Assign your result to: result_8
#   table_info = [
#     ('sales_transactions', transactions, 'transactionID'),
#     ('sales_customers', customers, 'customerID'),
#     ('sales_franchises', franchises, 'franchiseID'),
#     ('sales_suppliers', suppliers, <pk_col>),
#   ]
#   For each (name, df, pk):
#     row_count = df.count()
#     null_cols = [c for c in df.columns if df.filter(F.col(c).isNull()).count() > 0]
#     dup_pk = row_count - df.select(pk).distinct().count()
#   Then spark.createDataFrame(rows, schema)

result_8 = None  # replace this

In [ ]:
# ── Tests for Problem 8 ──────────────────────────────────────────
assert result_8 is not None, "result_8 is None — did you assign your DataFrame?"
assert hasattr(result_8, 'columns'), "result_8 must be a Spark DataFrame"
cols = [c.lower() for c in result_8.columns]
for col in ['table_name', 'row_count', 'null_columns', 'duplicate_pk_count']:
    assert col in cols, f"Missing column: {col}"
cnt = result_8.count()
assert cnt == 4, f"Expected exactly 4 rows (one per sales table), got {cnt}"
table_names = [r['table_name'] for r in result_8.select('table_name').collect()]
assert 'sales_transactions' in table_names, "Missing row for sales_transactions"
assert 'sales_customers' in table_names, "Missing row for sales_customers"
assert result_8.agg(F.min('row_count')).collect()[0][0] > 0, "row_count must be positive"
print(f"Problem 8 passed ✓  ({cnt} rows)")